# nablatensor — adjoint engine benchmark

One fixed workload (a 252-step down-and-in put, `fp32`, price + 5 Greeks from one
reverse sweep) run on **every engine the ServiceLoader reports as available**.
Each engine runs in its **own subprocess / JVM** — an isolated benchmark, and one
engine crashing can't take down the rest. Output: a result table and a
plain-text report to paste elsewhere.

Built for Google Colab with a **GPU runtime** (*Runtime ▸ Change runtime type ▸
T4 GPU*) so `cuda` is in the comparison. Runs anywhere; with no GPU it just
benchmarks the CPU-side engines.

In [ ]:
# --- Setup. On Colab: install JDK 25 + Maven, clone, build, install the bridge
#     (a few minutes, once per runtime). Elsewhere: use the local checkout. ---
import os, sys, subprocess

ON_COLAB = "google.colab" in sys.modules

if not ON_COLAB:
    PROJECT_ROOT = os.environ.get("NABLATENSOR_HOME")           # None -> bridge auto-finds
else:
    PROJECT_ROOT = "/content/nablatensor"
    JDK_HOME = "/opt/jdk-25"
    if not os.path.isdir(PROJECT_ROOT + "/.git"):
        _script = r'''
            set -eux
            if [ ! -x "$JDK_HOME/bin/java" ]; then
                curl -fsSL -o /tmp/jdk25.tgz \
                  "https://api.adoptium.net/v3/binary/latest/25/ga/linux/x64/jdk/hotspot/normal/eclipse"
                mkdir -p "$JDK_HOME"
                tar -xzf /tmp/jdk25.tgz -C "$JDK_HOME" --strip-components=1
            fi
            command -v mvn >/dev/null 2>&1 || { apt-get -qq update && apt-get -qq install -y maven; }
            [ -d "$PROJECT_ROOT/.git" ] || \
              git clone --depth 1 https://github.com/nablatensor-dev/nablatensor.git "$PROJECT_ROOT"
            cd "$PROJECT_ROOT"
            JAVA_HOME="$JDK_HOME" MAVEN_OPTS=--sun-misc-unsafe-memory-access=allow \
              mvn -q -T1C install -Dmaven.test.skip=true
            pip -q install ./python
        '''
        subprocess.run(["bash", "-c", _script], check=True,
                       env={**os.environ, "JDK_HOME": JDK_HOME, "PROJECT_ROOT": PROJECT_ROOT})
    os.environ["JAVA_HOME"] = JDK_HOME
    os.environ["PATH"] = JDK_HOME + "/bin:" + os.environ["PATH"]
    os.environ["LD_LIBRARY_PATH"] = "/usr/lib64-nvidia:" + os.environ.get("LD_LIBRARY_PATH", "")

PR_ARG = repr(PROJECT_ROOT) if PROJECT_ROOT else "None"
print("colab :", ON_COLAB, "| project root:", PROJECT_ROOT or "(auto)")

In [ ]:
# --- config -------------------------------------------------------------------
STEPS, SEED   = 252, 42
PARITY_PATHS  = 1_000_000       # every engine, for the price/greek comparison
REPEAT        = 3               # timed repeats for the throughput number
PER_ENGINE_TIMEOUT = 1800       # seconds per subprocess

# Engines never to launch. 'rocm' by default: HIP on some integrated / unofficial
# GPUs can hang or reset the machine, and Colab has no AMD GPU anyway. Clear the
# set (or drop a name) to include it; add names to exclude more.
SKIP_ENGINES = {"rocm"}
ONLY_ENGINES = set()           # if non-empty, benchmark just these

GPU_ENGINES = {"cuda", "vulkan", "rocm", "opencl"}
def perf_paths_for(name):
    if name in GPU_ENGINES:        return 20_000_000
    if name in ("simd", "cpu-jit"): return 4_000_000
    return 400_000                 # pure-scalar 'cpu'

def wants_fp32(e):
    # Request fp32 only where the engine advertises it; asking an fp64-only
    # engine for fp32 and letting it fall back has proved fragile.
    d = (e.get("describe") or "").lower()
    if "fp32" in d:
        return True
    if "fp64" in d:                # e.g. the scalar 'cpu' engine
        return False
    return e["name"] in GPU_ENGINES

def selected(name):
    if ONLY_ENGINES:
        return name in ONLY_ENGINES
    return name not in SKIP_ENGINES

WORKLOAD = f"down-and-in put, {STEPS} steps, fp32, price+5 greeks, seed {SEED}"
print("workload:", WORKLOAD)
print("parity  :", f"{PARITY_PATHS:,} paths (all engines)")
print("perf    : per-engine — gpu 20,000,000 | simd/cpu-jit 4,000,000 | cpu 400,000")
print("skip    :", ", ".join(sorted(SKIP_ENGINES)) or "(none)",
      "| only:", ", ".join(sorted(ONLY_ENGINES)) or "(all)")

In [ ]:
# --- the per-engine worker (runs in its own process / JVM) -------------------
import json, tempfile, textwrap

_WORKER_SRC = textwrap.dedent(r"""
    import sys, json, time
    eng          = sys.argv[1]
    parity_paths = int(sys.argv[2])
    perf_paths   = int(sys.argv[3])
    repeat       = int(sys.argv[4])
    fp32         = sys.argv[5] == "1"
    project_root = sys.argv[6] or None
    STEPS, SEED  = int(sys.argv[7]), int(sys.argv[8])

    out = {"engine": eng, "perf_paths": perf_paths,
           "parity_paths": parity_paths, "precision": "fp32" if fp32 else "fp64",
           "error": None}
    try:
        import nablatensor as nt
        nt.start(project_root=project_root)
        market = nt.EquityMarket(100.0, 100.0, 0.28, 0.03, 1.0)
        note = nt.ExoticProducts.barrier(
            nt.OptionType.PUT, nt.ExoticProducts.Barrier.DOWN_IN, 70.0, 1.0)

        b = nt.MonteCarlo.of(note).market(market).steps(STEPS).greeks().on(eng)
        if fp32:
            b = b.fp32()
        t = time.perf_counter(); mc = b.build(); out["build_s"] = time.perf_counter() - t
        out["nodes"] = int(mc.nodes())
        out["engine_actual"] = str(mc.engine())

        t = time.perf_counter(); p = mc.run(parity_paths, SEED)
        out["parity_s"] = time.perf_counter() - t
        out["price"] = float(p.price())
        out["delta"] = float(p.greeks().spot())

        best = kern = None
        for _ in range(repeat):
            t = time.perf_counter(); q = mc.run(perf_paths, SEED); dt = time.perf_counter() - t
            best = dt if best is None else min(best, dt)
            ks = float(q.seconds()); kern = ks if kern is None else min(kern, ks)
        out["warm_s"] = best
        out["kernel_s"] = kern
        out["mpps"] = perf_paths / best / 1e6 if best else None
        mc.close()
    except BaseException as ex:                       # noqa: BLE001
        out["error"] = f"{type(ex).__name__}: {str(ex)[:200]}"
    print("BENCH_JSON " + json.dumps(out), flush=True)
""")

_WORKER = os.path.join(tempfile.gettempdir(), "nt_bench_worker.py")
open(_WORKER, "w").write(_WORKER_SRC)

def run_worker(engine, parity, perf, repeat, fp32=True):
    cmd = [sys.executable, _WORKER, engine, str(parity), str(perf), str(repeat),
           "1" if fp32 else "0", PROJECT_ROOT or "", str(STEPS), str(SEED)]
    try:
        cp = subprocess.run(cmd, capture_output=True, text=True,
                            timeout=PER_ENGINE_TIMEOUT)
    except subprocess.TimeoutExpired:
        return {"engine": engine, "error": f"timeout > {PER_ENGINE_TIMEOUT}s"}
    for ln in cp.stdout.splitlines():
        if ln.startswith("BENCH_JSON "):
            return json.loads(ln[len("BENCH_JSON "):])
    tail = " / ".join((cp.stderr.strip().splitlines() or ["(no stderr)"])[-2:])
    return {"engine": engine,
            "error": f"process exited rc={cp.returncode} without a result :: {tail[:160]}"}

# --- discover engines (isolated: a crash here still leaves us the stdout) -----
_disc = ("import json, nablatensor as nt; nt.start(project_root=%s); "
         "print('ENG_JSON ' + json.dumps(nt.engines()))" % PR_ARG)
_cp = subprocess.run([sys.executable, "-c", _disc], capture_output=True, text=True, timeout=300)
engines = next((json.loads(l[len("ENG_JSON "):]) for l in _cp.stdout.splitlines()
                if l.startswith("ENG_JSON ")), None)
if engines is None:
    print("!! engine discovery produced no result; falling back to the known list")
    print(_cp.stderr[-400:])
    engines = [{"name": n, "priority": p, "available": None, "describe": "(probe unavailable)"}
               for n, p in [("cuda", 100), ("vulkan", 60), ("rocm", 55),
                            ("simd", 50), ("opencl", 45), ("cpu", 10), ("cpu-jit", 7)]]

print(f"{'engine':<10}{'prio':>5}{'ok':>5}   describe")
for e in engines:
    ok = {True: "ok", False: "--", None: "?"}[e["available"]]
    print(f"{e['name']:<10}{e['priority']:>5}{ok:>5}   {e['describe']}")

In [ ]:
# --- reference (fp64) + benchmark every engine ------------------------------
ref = {}
for cand in ("cpu", "simd", "cpu-jit"):
    if selected(cand) and any(e["name"] == cand and e["available"] for e in engines):
        print(f"reference: {cand} fp64, {PARITY_PATHS:,} paths ...")
        ref = run_worker(cand, PARITY_PATHS, PARITY_PATHS, 1, fp32=False)
        if not ref.get("error"):
            print(f"  price={ref['price']:.6f}  delta={ref['delta']:.6f}")
            break
        print("  failed:", ref["error"]); ref = {}
ref_price = ref.get("price")
ref_delta = ref.get("delta")

rows = []
for e in engines:
    name = e["name"]
    if e["available"] is False:
        rows.append({"engine": name, "prio": e["priority"], "skip": "not available"})
        continue
    if not selected(name):
        rows.append({"engine": name, "prio": e["priority"],
                     "skip": "skipped (config)"})
        print(f"\n>>> {name}  skipped by config")
        continue
    perf = perf_paths_for(name)
    fp32 = wants_fp32(e)
    print(f"\n>>> {name}  ({'fp32' if fp32 else 'fp64'}, parity {PARITY_PATHS:,} / "
          f"perf {perf:,} x{REPEAT}) ...", flush=True)
    r = run_worker(name, PARITY_PATHS, perf, REPEAT, fp32=fp32)
    r["prio"] = e["priority"]
    if r.get("error"):
        print("    ERROR:", r["error"])
    else:
        if ref_price:
            r["dprice"] = abs(r["price"] - ref_price) / abs(ref_price)
            r["ddelta"] = abs(r["delta"] - ref_delta) / abs(ref_delta)
        print(f"    build {r.get('build_s', float('nan')):.3f}s   "
              f"warm {r.get('warm_s', float('nan')):.3f}s   "
              f"{r.get('mpps') or 0:,.1f} Mpaths/s   price {r.get('price', float('nan')):.6f}")
    rows.append(r)
print("\nall engines done.")

In [ ]:
# --- result table ----------------------------------------------------------
def _g(x, spec, dash="-"):
    try:
        return format(x, spec) if x is not None else dash
    except (TypeError, ValueError):
        return dash

HDR = ["engine", "prec", "prio", "perf_paths", "build_s", "parity_s", "warm_s",
       "kern_s", "Mpaths/s", "price", "d_price", "d_delta", "note"]
W   = [10, 15, 4, 11, 8, 8, 8, 8, 9, 11, 8, 8, 40]

def _row(cells):
    return " ".join(str(c)[:wi].ljust(wi) for c, wi in zip(cells, W))

def table_lines():
    out = [_row(HDR), _row(["-" * wi for wi in W])]
    for r in rows:
        if r.get("skip"):
            out.append(_row([r["engine"], "", r.get("prio", ""), "", "", "", "",
                             "", "", "", "", "", r["skip"]]))
            continue
        out.append(_row([
            r.get("engine_actual", r["engine"]), r.get("precision", ""),
            r.get("prio", ""), _g(r.get("perf_paths"), ",d"),
            _g(r.get("build_s"), ".3f"), _g(r.get("parity_s"), ".3f"),
            _g(r.get("warm_s"), ".3f"), _g(r.get("kernel_s"), ".3f"),
            _g(r.get("mpps"), ",.1f"), _g(r.get("price"), ".6f"),
            _g(r.get("dprice"), ".1e"), _g(r.get("ddelta"), ".1e"),
            (r.get("error") or "")[:40],
        ]))
    return out

print("\n".join(table_lines()))

In [ ]:
# --- plain-ASCII report (copy/paste) -------------------------------------------
import datetime, platform

def _sh(cmd):
    try:
        return subprocess.run(cmd, shell=True, capture_output=True,
                              text=True, timeout=15).stdout.strip()
    except Exception:
        return ""

_cpu_model = ""
try:
    for _l in open("/proc/cpuinfo"):
        if _l.startswith("model name"):
            _cpu_model = _l.split(":", 1)[1].strip(); break
except Exception:
    pass
_gpu  = _sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader") or "none"
_java = _sh(f'"{os.environ.get("JAVA_HOME","")}/bin/java" -version 2>&1 | head -1') or "unknown"
_root = PROJECT_ROOT or "."
_sha  = _sh(f"git -C '{_root}' rev-parse --short HEAD") or _sh("git rev-parse --short HEAD")
_brn  = _sh(f"git -C '{_root}' rev-parse --abbrev-ref HEAD") or _sh("git rev-parse --abbrev-ref HEAD")

L = []
L.append("=" * 100)
L.append("NABLATENSOR ADJOINT ENGINE BENCHMARK")
L.append("=" * 100)
L.append(f"generated  : {datetime.datetime.now(datetime.timezone.utc):%Y-%m-%d %H:%M:%SZ}")
L.append(f"host       : {platform.platform()}")
L.append(f"cpu        : {_cpu_model or 'unknown'}  x{os.cpu_count()}")
L.append(f"gpu        : {_gpu}")
L.append(f"java       : {_java}")
L.append(f"nablatensor: {_sha or '?'} ({_brn or '?'})")
L.append(f"colab      : {ON_COLAB}")
L.append("-" * 100)
_nodes = next((r["nodes"] for r in rows if r.get("nodes")), ref.get("nodes"))
L.append(f"workload   : {WORKLOAD}"
         + (f"  ({_nodes} tape nodes)" if _nodes else ""))
L.append(f"parity     : {PARITY_PATHS:,} paths, all engines")
L.append(f"perf       : gpu 20,000,000 | simd/cpu-jit 4,000,000 | cpu 400,000 ; "
         f"warm_s = best of {REPEAT}")
L.append(f"isolation  : one subprocess + JVM per engine")
if SKIP_ENGINES or ONLY_ENGINES:
    L.append(f"config     : skip={sorted(SKIP_ENGINES) or '-'}  only={sorted(ONLY_ENGINES) or '-'}")
if ref_price is not None:
    L.append(f"reference  : {ref.get('engine')} fp64  price={ref_price:.6f}  delta={ref_delta:.6f}"
             f"   (d_price / d_delta = relative error vs this)")
L.append("-" * 100)
L.append("ENGINES DISCOVERED (priority order)")
for e in engines:
    ok = {True: "ok", False: "--", None: "? "}[e["available"]]
    L.append(f"  {ok:>2}  {e['name']:<9} p{e['priority']:<4} {e['describe']}")
L.append("-" * 100)
L.append("RESULTS")
L += table_lines()

_ok = [r for r in rows if r.get("mpps")]
if _ok:
    _fast = max(_ok, key=lambda r: r["mpps"])
    L.append("-" * 100)
    L.append(f"fastest    : {_fast.get('engine_actual', _fast['engine'])}  "
             f"{_fast['mpps']:,.1f} Mpaths/s  ({_fast['warm_s']:.3f}s / {_fast['perf_paths']:,} paths)")
    for r in sorted(_ok, key=lambda r: -r["mpps"]):
        L.append(f"  {r.get('engine_actual', r['engine']):<10} {r['mpps']:>10,.1f} Mpaths/s   "
                 f"{_fast['mpps'] / r['mpps']:>6.1f}x")

_err = [(r["engine"], r["error"]) for r in rows if r.get("error")]
if _err:
    L.append("-" * 100)
    L.append("ERRORS / CRASHES")
    for n, m in _err:
        L.append(f"  {n:<10} {m}")
L.append("=" * 100)

report = "\n".join(L)
_out = "/content/nablatensor-engine-benchmark.txt" if ON_COLAB else "nablatensor-engine-benchmark.txt"
open(_out, "w").write(report + "\n")

print("----- COPY BELOW " + "-" * 82)
print(report)
print("----- COPY ABOVE " + "-" * 82)
print(f"\n(also written to {_out})")